# 04 · BKT: the first knowledge tracer
**Kuizi brain walkthrough, notebook 4 of 10**

**Bayesian Knowledge Tracing** (BKT, 1995) imagines a hidden switch for each concept: *known* or
*not known*. We never see the switch, only answers. Four numbers per concept describe it:

| Parameter | Meaning |
| --- | --- |
| P(init) | chance the student already knows the concept at the start |
| P(learn) | chance of learning it after each practice |
| P(guess) | chance of a right answer without knowing (lucky) |
| P(slip) | chance of a wrong answer while knowing (careless) |

After every answer, **Bayes' rule** updates the belief that the switch is on, then the student
may learn. BKT is simple and explainable; it is one of the baselines in the T040 benchmark.

<div dir="rtl" lang="ar">

## ٠٤ · BKT: أوّل متتبّع للمعرفة

<b>تتبّع المعرفة البايزي</b> (BKT) يتخيّل مفتاحًا مخفيًا لكل مفهوم: <b>معروف</b> أو <b>غير معروف</b>. لا نرى المفتاح أبدًا،
بل نرى الإجابات فقط. أربعة أرقام لكل مفهوم: احتمال المعرفة في البداية، احتمال التعلّم بعد كل تمرين،
احتمال التخمين الصحيح دون معرفة، واحتمال الخطأ رغم المعرفة (السهو).

بعد كل إجابة تحدّث <b>قاعدة بايز</b> درجة اعتقادنا بأن المفتاح مشغّل، ثم قد يتعلّم التلميذ. BKT بسيط وقابل للشرح،
وهو أحد نماذج المقارنة في T040.

</div>

In [ ]:
# Concept: one Bayes update by hand · Production: T040
p_known, guess, slip, learn = 0.30, 0.20, 0.10, 0.15
for answer in [1, 1, 0, 1, 1]:
    p_right = p_known * (1 - slip) + (1 - p_known) * guess
    if answer == 1:
        posterior = p_known * (1 - slip) / p_right
    else:
        posterior = p_known * slip / (1 - p_right)
    after_learning = posterior + (1 - posterior) * learn
    print(f"predicted P(right)={p_right:.2f}  answer={'right' if answer else 'wrong'}  "
          f"P(known) {p_known:.2f} -> {posterior:.2f} -> after practice {after_learning:.2f}")
    p_known = after_learning

Notice: a wrong answer lowers the belief sharply, but not to zero, because people slip. Two right
answers in a row push it up quickly, but not to 100 %, because people guess.

Now we let the computer find the four numbers for each of the 145 ASSISTments concepts by
**gradient descent**: try numbers, measure how badly they predict the answers (the **loss**),
nudge them to do better, repeat.

<div dir="rtl" lang="ar">

لاحظ: الإجابة الخاطئة تخفض الاعتقاد بقوّة لكن ليس إلى الصفر، لأن الناس يسهون. وإجابتان صحيحتان متتاليتان ترفعانه
بسرعة لكن ليس إلى 100%، لأن الناس يخمّنون.

الآن نترك الحاسوب يجد الأرقام الأربعة لكل مفهوم بـ<b>الانحدار التدرّجي</b>: نجرّب أرقامًا، نقيس سوء تنبّؤها (<b>الخسارة</b>)،
نعدّلها قليلًا لتتحسّن، ونكرّر.

</div>

In [ ]:
# Concept: training BKT on real answers · Production: T040
import numpy as np, pandas as pd, torch
from kuizi_brain import data as D, models as M, metrics as X
df = D.load_assist2009()
train, test = D.split_by_student(df)
w_train, w_test = M.make_windows(train), M.make_windows(test)
bkt = M.BKT(df.kc.max() + 1)
M.fit(bkt, w_train, epochs=4, lr=0.05)
y, p = M.predict(bkt, w_test)
pd.DataFrame([X.report("BKT", y, p)])

In [ ]:
# Concept: reading the learned parameters · Production: T040, T049
init, learn, guess, slip = [t.detach().numpy() for t in bkt.params()]
names = df.groupby("kc").kc_name.first()
top = df.kc.value_counts().index[:8]
pd.DataFrame({"concept": names[top].values, "P(init)": init[top + 1], "P(learn)": learn[top + 1],
              "P(guess)": guess[top + 1], "P(slip)": slip[top + 1]}).round(2)

In [ ]:
# Concept: tracing one student's belief over time · Production: T040, T049
import matplotlib.pyplot as plt
kc = top[0]
student = test[test.kc == kc].student.value_counts().index[0]
hist = test[test.student == student]
w = M.make_windows(hist, max_len=len(hist))
with torch.no_grad():
    p_right = torch.sigmoid(bkt(w))[0].numpy()
g, s = guess[kc + 1], slip[kc + 1]
mask = hist.kc.values == kc
p_known = (p_right[mask] - g) / (1 - s - g)
answers = hist.correct.values[mask]
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(range(1, len(p_known) + 1), p_known, "-", label="P(known) before each answer")
ax.scatter(np.flatnonzero(answers == 1) + 1, np.ones((answers == 1).sum()) * 1.03, marker="o", label="right answer")
ax.scatter(np.flatnonzero(answers == 0) + 1, np.ones((answers == 0).sum()) * -0.03, marker="x", label="wrong answer")
ax.set_ylim(-0.1, 1.1); ax.set_xlabel(f"practice on '{names[kc]}'"); ax.legend(loc="center right")
ax.set_title("BKT's belief rises with right answers and falls with wrong ones")
plt.show()

### Limits of BKT
- Each concept lives alone: knowing *Dérivées* tells BKT nothing about *Dipôle RC*.
- No forgetting, no time, no hints, no item difficulty: every item of a concept looks the same.
- Its probabilities are often badly **calibrated** (see the ECE number above; notebook 8 explains it).

The next models read the whole history at once to fix these limits.

### Remember
- BKT = a hidden known/not-known switch per concept, updated with Bayes' rule after each answer.
- Four interpretable numbers per concept: init, learn, guess, slip.
- A useful, explainable baseline, but too simple to be Kuizi's brain.

<div dir="rtl" lang="ar">

### حدود BKT
- كل مفهوم يعيش وحده: معرفة الاشتقاق لا تخبر BKT بشيء عن ثنائي القطب RC.
- لا نسيان، ولا زمن، ولا تلميحات، ولا صعوبة للتمارين: كل تمارين المفهوم متشابهة عنده.
- احتمالاته غالبًا سيّئة <b>المعايرة</b> (انظر رقم ECE أعلاه؛ الدفتر ٨ يشرحه).

### تذكّر
- BKT = مفتاح مخفي (معروف/غير معروف) لكل مفهوم، يُحدَّث بقاعدة بايز بعد كل إجابة.
- أربعة أرقام مفهومة لكل مفهوم: البداية، التعلّم، التخمين، السهو.
- نموذج مقارنة مفيد وقابل للشرح، لكنه أبسط من أن يكون دماغ كويزي.

</div>